# Inference: Base, Teacher, Student

Runs all three models on 60 German customer-support queries and saves the raw responses.<br>
Results are printed directly as notebook output. Optionally, they can be saved locally via the switch in Section 4.<br>
<br>
**Pipeline position:** 01 Fine-Tuning → 02 Data Generation → 03 Student Distillation → `[04 Inference]` → 05 Evaluation<br>
**Strong GPU required.** This notebook was developed on a Kaggle T4 (16 GB VRAM).<br>
[![Open Notebook in Kaggle](https://img.shields.io/badge/Open%20Notebook%20in-Kaggle-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white)](https://www.kaggle.com/code/dennisfeyerabend/04-inference)

## 1. Setup

Install dependencies and check GPU.  
Local users: skip the pip cell — install via `pip install -r requirements.txt` instead.  
This notebook is inference-only — no training libraries needed.

In [1]:
%%capture
!pip install -q --upgrade unsloth transformers peft bitsandbytes accelerate python-dotenv

In [2]:
import gc
import json
import time
import os
import torch

print(f"PyTorch version:  {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
    print(f"VRAM:             {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version:  2.10.0+cu128
CUDA available:   True
GPU:              Tesla T4
VRAM:             15.6 GB


### **Models evaluated in this notebook**

Three models are compared — loaded one at a time, in this order:

- **Base** — `Qwen/Qwen2.5-1.5B-Instruct` (no fine-tuning). The out-of-the-box 1.5B model.
  Serves as the baseline: shows what the student model looks like before distillation.
- **Teacher** — `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` with LoRA adapter `Feyerade/german-support-qwen-lora-adapter`.   
  The 3B model fine-tuned on synthetic German support data in Notebook 01.<br>
  Sets the target style the student is trained to match.<br>
- **Student** — `Feyerade/german-support-student-1.5b-distilled`.
  The 1.5B model distilled from teacher outputs in Notebook 03. Standalone merged checkpoint — no adapter needed.

Raw responses will be printed out directly after generating.   
Alternatively, responses can be saved to Kaggle.

In [3]:
BASE_MODEL_ID    = "Qwen/Qwen2.5-1.5B-Instruct"
TEACHER_BASE_ID  = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
TEACHER_ADAPTER  = "Feyerade/german-support-qwen-lora-adapter"
STUDENT_MODEL_ID = "Feyerade/german-support-student-1.5b-distilled"

print(f"Base model:      {BASE_MODEL_ID}")
print(f"Teacher base:    {TEACHER_BASE_ID}")
print(f"Teacher adapter: {TEACHER_ADAPTER}")
print(f"Student model:   {STUDENT_MODEL_ID}")

Base model:      Qwen/Qwen2.5-1.5B-Instruct
Teacher base:    unsloth/Qwen2.5-3B-Instruct-bnb-4bit
Teacher adapter: Feyerade/german-support-qwen-lora-adapter
Student model:   Feyerade/german-support-student-1.5b-distilled


## 2. Queries, Parameters, and Helpers

Defines the 60 evaluation queries, generation parameters, and the helper functions used in Section 4.

In [4]:
test_queries = [
  {"id": "q01", "query": "Guten Tag, ich wollte mich erkundigen, ob Sie auch in die Schweiz liefern. Falls ja, wuerde ich gerne wissen wie hoch die Versandkosten ausfallen.", "category": "Orders & Shipping", "sentence_count": 2, "mood": "polite"},
  {"id": "q02", "query": "Koennten Sie mir bitte mitteilen, ob beim Versand eine Geschenkverpackung mit Karte moeglich ist.", "category": "Orders & Shipping", "sentence_count": 1, "mood": "polite"},
  {"id": "q03", "query": "Ich ziehe in zwei Wochen um und wuerde meine Bestellung erst danach erhalten wollen. Gibt es eine Moeglichkeit ein Wunschdatum fuer die Lieferung anzugeben? Andernfalls warte ich mit dem Bestellen bis nach dem Umzug.", "category": "Orders & Shipping", "sentence_count": 3, "mood": "polite"},
  {"id": "q04", "query": "Ich wuerde meine Bestellung gerne an eine DHL-Packstation liefern lassen. Wo trage ich die Postnummer im Bestellprozess ein?", "category": "Orders & Shipping", "sentence_count": 2, "mood": "polite"},
  {"id": "q05", "query": "Mein Paket hat seit Donnerstag den Status 'in Zustellung', aber bei mir kommt nichts an.", "category": "Orders & Shipping", "sentence_count": 1, "mood": "frustrated"},
  {"id": "q06", "query": "Ich habe extra fuer Expressversand acht Euro bezahlt und die Sendung kommt jetzt spaeter als bei Standardversand. Was soll das.", "category": "Orders & Shipping", "sentence_count": 2, "mood": "frustrated"},
  {"id": "q07", "query": "Der Bote hat mein Paket einfach offen im Hausflur abgestellt ohne zu klingeln. Zwei Artikel fehlen jetzt aus der Sendung. Das ist mittlerweile das dritte Mal in diesem Jahr.", "category": "Orders & Shipping", "sentence_count": 3, "mood": "frustrated"},
  {"id": "q08", "query": "Beim Checkout standen zwei Tage Lieferzeit, jetzt nach der Bestellung sind es ploetzlich zehn Werktage. So macht man keine Kunden gluecklich.", "category": "Orders & Shipping", "sentence_count": 2, "mood": "frustrated"},
  {"id": "q09", "query": "Versand nach Belgien moeglich", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise"},
  {"id": "q10", "query": "Bestelle Sperrgut wird das normal verschickt", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise"},
  {"id": "q11", "query": "Samstagszustellung anbieten ja oder nein", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise"},
  {"id": "q12", "query": "Wie kann ich den Versand nachtraeglich beschleunigen", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise"},

  {"id": "q13", "query": "Sehr geehrte Damen und Herren, leider ist mein gestern geliefertes Geraet bereits beim Auspacken zerbrochen. Wie ist hier das weitere Vorgehen?", "category": "Returns & Complaints", "sentence_count": 2, "mood": "polite"},
  {"id": "q14", "query": "Ich habe vor einem Jahr einen Akku-Staubsauger bei Ihnen gekauft. Der Akku haelt mittlerweile nur noch 5 Minuten statt 45. Faellt das noch unter die gesetzliche Gewaehrleistung?", "category": "Returns & Complaints", "sentence_count": 3, "mood": "polite"},
  {"id": "q15", "query": "Ist es moeglich, die Rueckgabefrist um eine Woche zu verlaengern, da ich wegen eines Klinikaufenthalts erst jetzt zur Pruefung der Ware gekommen bin.", "category": "Returns & Complaints", "sentence_count": 1, "mood": "polite"},
  {"id": "q16", "query": "Mein Sohn hat mir eine Kaffeemaschine geschenkt, die jetzt nach acht Monaten nicht mehr funktioniert. Kann ich die Garantie auch ohne meinen eigenen Kaufbeleg in Anspruch nehmen?", "category": "Returns & Complaints", "sentence_count": 2, "mood": "polite"},
  {"id": "q17", "query": "Ich warte seit fuenf Tagen auf das Ruecksendeetikett, das mir per E-Mail zugeschickt werden sollte und nichts kommt an.", "category": "Returns & Complaints", "sentence_count": 1, "mood": "frustrated"},
  {"id": "q18", "query": "Drei von vier bestellten Weinglaesern aus derselben Lieferung sind angeschlagen. Ist das bei euch normal.", "category": "Returns & Complaints", "sentence_count": 2, "mood": "frustrated"},
  {"id": "q19", "query": "Mein Saugroboter ist nach genau 15 Monaten kaputtgegangen. In der Werbung stand 24 Monate Herstellergarantie aber jetzt weigert sich euer Service. Was soll dieser Werbeschwindel.", "category": "Returns & Complaints", "sentence_count": 3, "mood": "frustrated"},
  {"id": "q20", "query": "Die bestellte Hose ist viel kleiner ausgefallen als die angegebene Groesse vermuten laesst. Ich verlange eine Rueckerstattung der Versandkosten zusaetzlich zur Rueckzahlung.", "category": "Returns & Complaints", "sentence_count": 2, "mood": "frustrated"},
  {"id": "q21", "query": "Bekomme Erstattung auf andere Karte", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise"},
  {"id": "q22", "query": "Wie reparieren statt tauschen", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise"},
  {"id": "q23", "query": "Karton schon weg darf trotzdem zurueck", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise"},
  {"id": "q24", "query": "Garantieverlaengerung jetzt noch dazukaufen", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise"},

  {"id": "q25", "query": "Guten Tag, ich wuerde gerne kuenftig per SEPA-Lastschrift zahlen anstatt per Kreditkarte. Wie richte ich das in meinem Konto ein?", "category": "Payment & Billing", "sentence_count": 2, "mood": "polite"},
  {"id": "q26", "query": "Ich wollte fragen, ob es einen Studentenrabatt gibt fuer Kunden mit gueltigem Studentenausweis.", "category": "Payment & Billing", "sentence_count": 1, "mood": "polite"},
  {"id": "q27", "query": "Ich werde naechste Woche 30 und habe noch nie meinen Geburtstagsrabatt erhalten. Ich erhalte zwar regelmaessig den Newsletter aber kein Geburtstagscode kam je an. Faellt das auf?", "category": "Payment & Billing", "sentence_count": 3, "mood": "polite"},
  {"id": "q28", "query": "Ich wuerde gerne eine groessere Anschaffung ueber 800 Euro in mehreren Raten zahlen. Welche Anbieter unterstuetzen Sie hier?", "category": "Payment & Billing", "sentence_count": 2, "mood": "polite"},
  {"id": "q29", "query": "Meine Rueckerstattung ueber 89 Euro ist seit drei Wochen unterwegs und auf meinem Konto kommt einfach nichts an.", "category": "Payment & Billing", "sentence_count": 1, "mood": "frustrated"},
  {"id": "q30", "query": "Der Rabattcode aus eurem Newsletter geht angeblich erst ab 50 Euro Mindestbestellwert. Davon stand in der Mail null.", "category": "Payment & Billing", "sentence_count": 2, "mood": "frustrated"},
  {"id": "q31", "query": "Mein Abonnement laeuft trotz schriftlicher Kuendigung naechsten Monat einfach weiter. Ich habe drei E-Mails geschrieben in den letzten 14 Tagen. Keine einzige Antwort von euch.", "category": "Payment & Billing", "sentence_count": 3, "mood": "frustrated"},
  {"id": "q32", "query": "Mein Konto wurde belastet, obwohl ich die Bestellung am gleichen Tag noch storniert habe. Wann bekomme ich das Geld zurueck.", "category": "Payment & Billing", "sentence_count": 2, "mood": "frustrated"},
  {"id": "q33", "query": "Klarna Ratenzahlung verfuegbar", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise"},
  {"id": "q34", "query": "Brauche Rechnung mit Firma als Empfaenger", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise"},
  {"id": "q35", "query": "Abo nur pausieren geht das", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise"},
  {"id": "q36", "query": "Bonuspunkte einsetzen wie", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise"},

  {"id": "q37", "query": "Mein Vater ist leider letzten Monat verstorben und hatte ein aktives Konto bei Ihnen mit einem laufenden Abonnement. Wie kann ich das Konto schliessen und das Restguthaben sichern? Ich habe seine Sterbeurkunde vorliegen.", "category": "Account & Security", "sentence_count": 3, "mood": "polite"},
  {"id": "q38", "query": "Ich habe versehentlich zwei Konten unter unterschiedlichen E-Mail-Adressen angelegt. Ist es moeglich, beide zu einem zusammenzufuehren?", "category": "Account & Security", "sentence_count": 2, "mood": "polite"},
  {"id": "q39", "query": "Ich moechte gerne eine Auskunft ueber alle bei Ihnen gespeicherten persoenlichen Daten anfordern. Welcher Ansprechpartner ist hierfuer zustaendig?", "category": "Account & Security", "sentence_count": 2, "mood": "polite"},
  {"id": "q40", "query": "Koennten Sie mir mitteilen, wo ich in meinem Konto die Liste der aktiven Sitzungen einsehen kann.", "category": "Account & Security", "sentence_count": 1, "mood": "polite"},
  {"id": "q41", "query": "Seit dem letzten App-Update muss ich mich jedes Mal komplett neu einloggen und das nervt extrem.", "category": "Account & Security", "sentence_count": 1, "mood": "frustrated"},
  {"id": "q42", "query": "Ihr habt mir wegen verdaechtiger Aktivitaet das Konto gesperrt obwohl ich nur im Urlaub in Italien war. So behandelt man keine langjaehrigen Kunden.", "category": "Account & Security", "sentence_count": 2, "mood": "frustrated"},
  {"id": "q43", "query": "Ich bekomme seit Wochen Spam-Mails an meine bei Euch hinterlegte Adresse. Die wird sonst nirgendwo verwendet. Ich vermute ein Datenleck bei Euch.", "category": "Account & Security", "sentence_count": 3, "mood": "frustrated"},
  {"id": "q44", "query": "Ich versuche seit gestern den Bestaetigungscode fuer die Zwei-Faktor-Anmeldung zu erhalten und die SMS kommt einfach nie an. Mein Empfang ist voll und andere SMS gehen problemlos durch.", "category": "Account & Security", "sentence_count": 2, "mood": "frustrated"},
  {"id": "q45", "query": "Backup-Codes neu generieren wie", "category": "Account & Security", "sentence_count": 1, "mood": "concise"},
  {"id": "q46", "query": "Login mit Apple-ID moeglich", "category": "Account & Security", "sentence_count": 1, "mood": "concise"},
  {"id": "q47", "query": "Sicherheitsfrage vergessen", "category": "Account & Security", "sentence_count": 1, "mood": "concise"},
  {"id": "q48", "query": "2FA per App statt SMS einrichten", "category": "Account & Security", "sentence_count": 1, "mood": "concise"},

  {"id": "q49", "query": "Wenn ich im Shop nach Kategorien filtere, werden mir trotzdem Produkte aus anderen Bereichen angezeigt. Liegt das an einem Fehler oder mache ich etwas falsch?", "category": "Technical & Other", "sentence_count": 2, "mood": "polite"},
  {"id": "q50", "query": "Auf der Produktseite laden seit gestern keine Bilder mehr und ich kann gar nicht erkennen was ich kaufe.", "category": "Technical & Other", "sentence_count": 1, "mood": "frustrated"},
  {"id": "q51", "query": "App auf Englisch umstellen geht das", "category": "Technical & Other", "sentence_count": 1, "mood": "concise"},
  {"id": "q52", "query": "Ich erhalte gefuehlt fuenf E-Mails pro Woche von Ihnen, was mir ehrlich gesagt zu viel ist. Gibt es eine Moeglichkeit, die Frequenz zu reduzieren ohne mich ganz abzumelden?", "category": "Technical & Other", "sentence_count": 2, "mood": "polite"},
  {"id": "q53", "query": "Ich habe Euren Newsletter speziell fuer Kindermode abonniert und bekomme stattdessen seit Wochen nur Werbung fuer Herrenrasierer.", "category": "Technical & Other", "sentence_count": 1, "mood": "frustrated"},
  {"id": "q54", "query": "Newsletter nur auf Deutsch bitte", "category": "Technical & Other", "sentence_count": 1, "mood": "concise"},
  {"id": "q55", "query": "Ich habe vor kurzem ein Hemd bei Ihnen gekauft und moechte gerne eine ausfuehrliche Bewertung schreiben. Dabei wuerde ich gerne ein Foto vom Produkt hinzufuegen, weiss aber nicht wie. Koennten Sie mir den Weg dahin beschreiben?", "category": "Technical & Other", "sentence_count": 3, "mood": "polite"},
  {"id": "q56", "query": "Ich habe eine ehrliche 2-Sterne-Bewertung abgegeben und der Verkaeufer antwortet darunter mit persoenlichen Beleidigungen. Wieso wird so etwas nicht moderiert.", "category": "Technical & Other", "sentence_count": 2, "mood": "frustrated"},
  {"id": "q57", "query": "Bewertung anonym moeglich", "category": "Technical & Other", "sentence_count": 1, "mood": "concise"},
  {"id": "q58", "query": "Mich wuerde interessieren, ob man mehrere Freunde gleichzeitig ueber einen Sammel-Empfehlungslink einladen kann. Das waere praktisch fuer unseren Klassenchat.", "category": "Technical & Other", "sentence_count": 2, "mood": "polite"},
  {"id": "q59", "query": "Mein Bruder hat mit meinem Empfehlungslink bestellt. Er hat die fuenf Euro Willkommensrabatt bekommen, ich aber bis heute keine Gutschrift. Das fuehlt sich nach Bauernfaengerei an.", "category": "Technical & Other", "sentence_count": 3, "mood": "frustrated"},
  {"id": "q60", "query": "Empfehlungslink wo finden in der App", "category": "Technical & Other", "sentence_count": 1, "mood": "concise"}
]

print(f"Queries loaded: {len(test_queries)}")

Queries loaded: 60


**Generation parameters**

All three models use identical settings so that any differences in output are due to the models themselves, not the sampling configuration.   

`max_new_tokens = 384` is set slightly higher than the 256 used during data generation in Notebook 02.  
The judge in Notebook 05 scores whether a response *ends* with a closing offer to help — a response truncated mid-sentence automatically loses that point regardless of model quality.    
384 tokens gives enough headroom for even verbose Teacher responses to complete naturally, while keeping per-query inference time acceptable on a T4.   

In [5]:
SYSTEM_PROMPT = (
    "Du bist ein professioneller Kundenservice-Mitarbeiter. "
    "Antworte freundlich, loesungsorientiert und auf Deutsch. "
    "Halte deine Antworten unter 150 Woertern."
)

GEN_PARAMS = {
    "max_new_tokens": 384,
    "temperature":    0.7,
    "top_p":          0.9,
    "do_sample":      True,
}

In [6]:
def run_inference(model, tokenizer, query_text, query_idx):
    """
    Generate one response for a single query.

    Returns:
        response_text       (str)   — decoded model output, prompt stripped
        generated_tokens    (int)   — number of new tokens produced
        generation_time_sec (float) — wall-clock seconds for model.generate()
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": query_text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    prompt_len = inputs["input_ids"].shape[1]

    torch.manual_seed(42 + query_idx)

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **GEN_PARAMS,
            pad_token_id=tokenizer.pad_token_id,
        )
    generation_time_sec = time.time() - t0

    generated_ids    = outputs[0][prompt_len:]
    response_text    = tokenizer.decode(generated_ids, skip_special_tokens=True)
    generated_tokens = len(generated_ids)

    return response_text, generated_tokens, generation_time_sec


def reset_vram():
    """
    Hard VRAM reset between model swaps.

    Ensures torch.cuda.max_memory_allocated() starts from zero for each model,
    so peak VRAM measurements are not contaminated by the previous model.
    """
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"VRAM reset — allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3. Sequential Inference

Runs all three models on the 60 evaluation queries in order: Base → Teacher → Student.

Each model follows the same sequence:
- Hard VRAM reset so peak memory measurements are clean and independent
- Load model
- One warmup generation (discarded) to absorb CUDA kernel compilation overhead
- 60 timed inference calls — responses printed inline
- Peak VRAM recorded
- Model deleted and VRAM reset before the next load

All three models use identical generation parameters (defined in Section 2).

In [7]:
from unsloth import FastLanguageModel
from peft import PeftModel
import warnings
import transformers

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

base_results    = []
teacher_results = []
student_results = []
vram_measurements = []

# Set to False to skip a model — useful when re-running one model
# without waiting for all three to complete again
RUN_BASE    = True
RUN_TEACHER = True
RUN_STUDENT = True

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### 3.1 Base — `Qwen2.5-1.5B-Instruct` (no fine-tuning)

Baseline: the out-of-the-box 1.5B model before any training.   
Shows what the student model gains from distillation.   

In [8]:
if RUN_BASE:
    # --- Load ---
    reset_vram()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    print("Base model ready.\n")

    # --- Warmup ---
    run_inference(model, tokenizer, "Hallo", -1)
    print("Warmup done. Starting inference...\n")

    # --- Inference ---
    for i, entry in enumerate(test_queries):
        response_text, generated_tokens, generation_time_sec = run_inference(
            model, tokenizer, entry["query"], i
        )
        tokens_per_sec = generated_tokens / generation_time_sec
        word_count     = len(response_text.split())

        base_results.append({
            "query_id":            entry["id"],
            "query_category":      entry["category"],
            "query_mood":          entry["mood"],
            "sentence_count":      entry["sentence_count"],
            "query_text":          entry["query"],
            "response_text":       response_text,
            "generated_tokens":    generated_tokens,
            "generation_time_sec": round(generation_time_sec, 3),
            "tokens_per_sec":      round(tokens_per_sec, 1),
            "word_count":          word_count,
        })

        print(f"{'='*60}")
        print(f"[{entry['id']}] {entry['query']}")
        print(f"{'='*60}")
        print(response_text)
        print(f"\n→ {generated_tokens} tok | {generation_time_sec:.1f}s | {tokens_per_sec:.1f} tok/s | {word_count} words\n")

    # --- VRAM ---
    vram_peak_gb = torch.cuda.max_memory_allocated() / 1e9
    vram_measurements.append({"model_label": "base", "vram_peak_gb": round(vram_peak_gb, 2)})
    print(f"Peak VRAM (base): {vram_peak_gb:.2f} GB")

    # --- Unload ---
    del model
    reset_vram()

VRAM reset — allocated: 0.01 GB
==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model ready.

Warmup done. Starting inference...

[q01] Guten Tag, ich wollte mich erkundigen, ob Sie auch in die Schweiz liefern. Falls ja, wuerde ich gerne wissen wie hoch die Versandkosten ausfallen.
Hallo! Ja, wir bieten unseren Kundenservice auch an der Schweiz an. Die Versandkosten variieren je nach dem gewünschten Versandgewicht und der Art des Pakets. Um eine genauere Information zu erhalten, bitte uns einen kurzen Lebenslauf oder ein Bild per E-Mail geben. Wir freuen uns darauf, Ihnen bei Ihren Anliegen behilflich zu sein.

→ 84 tok | 3.4s | 24.6 tok/s | 52 words

[q02] Koennten Sie mir bitte mitteilen, ob beim Versand eine Geschenkverpackung mit Karte moeglich ist.
Natürlich, bei unserer Lieferung könnt ihr die Geschenkverpackung mit einem Kärtchen kombinieren. Wir freuen uns darauf, dass ihr diesen Vorschlag akzeptiert! Wenn es euch nicht passt, könnt ihr un

### 3.2 Teacher — `Qwen2.5-3B-Instruct` + LoRA adapter

The fine-tuned 3B model from Notebook 01. Loaded in two steps: the quantised base first, then the LoRA adapter applied on top via PEFT. This is the target style the student was trained to imitate.

In [9]:
if RUN_TEACHER:
    # --- Load ---
    reset_vram()
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=TEACHER_BASE_ID,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    model = PeftModel.from_pretrained(base_model, TEACHER_ADAPTER)
    FastLanguageModel.for_inference(model)
    print("Teacher model ready.\n")

    # --- Warmup ---
    run_inference(model, tokenizer, "Hallo", -1)
    print("Warmup done. Starting inference...\n")

    # --- Inference ---
    for i, entry in enumerate(test_queries):
        response_text, generated_tokens, generation_time_sec = run_inference(
            model, tokenizer, entry["query"], i
        )
        tokens_per_sec = generated_tokens / generation_time_sec
        word_count     = len(response_text.split())

        teacher_results.append({
            "query_id":            entry["id"],
            "query_category":      entry["category"],
            "query_mood":          entry["mood"],
            "sentence_count":      entry["sentence_count"],
            "query_text":          entry["query"],
            "response_text":       response_text,
            "generated_tokens":    generated_tokens,
            "generation_time_sec": round(generation_time_sec, 3),
            "tokens_per_sec":      round(tokens_per_sec, 1),
            "word_count":          word_count,
        })

        print(f"{'='*60}")
        print(f"[{entry['id']}] {entry['query']}")
        print(f"{'='*60}")
        print(response_text)
        print(f"\n→ {generated_tokens} tok | {generation_time_sec:.1f}s | {tokens_per_sec:.1f} tok/s | {word_count} words\n")

    # --- VRAM ---
    vram_peak_gb = torch.cuda.max_memory_allocated() / 1e9
    vram_measurements.append({"model_label": "teacher", "vram_peak_gb": round(vram_peak_gb, 2)})
    print(f"Peak VRAM (teacher): {vram_peak_gb:.2f} GB")

    # --- Unload ---
    del model, base_model
    reset_vram()

VRAM reset — allocated: 0.01 GB
==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Teacher model ready.

Warmup done. Starting inference...

[q01] Guten Tag, ich wollte mich erkundigen, ob Sie auch in die Schweiz liefern. Falls ja, wuerde ich gerne wissen wie hoch die Versandkosten ausfallen.
Hi! Natürlich liefern wir in die Schweiz. Die Preise gehen von 14€ ab einem Gewicht von 0,5kg. Für Gewichte über 30kg haben wir einen Preis von 7€. Bitte beachten Sie: Die genaue Versandkostenstelle wird noch beim Senden angezeigt.

→ 72 tok | 5.5s | 13.2 tok/s | 38 words

[q02] Koennten Sie mir bitte mitteilen, ob beim Versand eine Geschenkverpackung mit Karte moeglich ist.
Selbstverstaendlich! Wir fuerkenfen gerne eine Geschenkverpackung mit einer Begleitkuenege an Sie. Bitte melden Sie sich bei uns nach dem Senddatum, den wir ankreisen.

→ 49 tok | 3.7s | 13.3 tok/s | 23 words

[q03] Ich ziehe in zwei Wochen um und wuerde meine Bestellung erst danach erhalten wollen. Gibt e

### 3.3 Student — `german-support-student-1.5b-distilled`   

The distilled 1.5B model from Notebook 03. Saved as a merged checkpoint — the LoRA adapter weights were folded into the base weights before upload, so no PEFT loading step is needed.   
Loaded identically to the base model.

In [10]:
if RUN_STUDENT:
    # --- Load ---
    reset_vram()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=STUDENT_MODEL_ID,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    print("Student model ready.\n")

    # --- Warmup ---
    run_inference(model, tokenizer, "Hallo", -1)
    print("Warmup done. Starting inference...\n")

    # --- Inference ---
    for i, entry in enumerate(test_queries):
        response_text, generated_tokens, generation_time_sec = run_inference(
            model, tokenizer, entry["query"], i
        )
        tokens_per_sec = generated_tokens / generation_time_sec
        word_count     = len(response_text.split())

        student_results.append({
            "query_id":            entry["id"],
            "query_category":      entry["category"],
            "query_mood":          entry["mood"],
            "sentence_count":      entry["sentence_count"],
            "query_text":          entry["query"],
            "response_text":       response_text,
            "generated_tokens":    generated_tokens,
            "generation_time_sec": round(generation_time_sec, 3),
            "tokens_per_sec":      round(tokens_per_sec, 1),
            "word_count":          word_count,
        })

        print(f"{'='*60}")
        print(f"[{entry['id']}] {entry['query']}")
        print(f"{'='*60}")
        print(response_text)
        print(f"\n→ {generated_tokens} tok | {generation_time_sec:.1f}s | {tokens_per_sec:.1f} tok/s | {word_count} words\n")

    # --- VRAM ---
    vram_peak_gb = torch.cuda.max_memory_allocated() / 1e9
    vram_measurements.append({"model_label": "student", "vram_peak_gb": round(vram_peak_gb, 2)})
    print(f"Peak VRAM (student): {vram_peak_gb:.2f} GB")

    # --- Unload ---
    del model
    reset_vram()

VRAM reset — allocated: 0.01 GB
==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Student model ready.

Warmup done. Starting inference...

[q01] Guten Tag, ich wollte mich erkundigen, ob Sie auch in die Schweiz liefern. Falls ja, wuerde ich gerne wissen wie hoch die Versandkosten ausfallen.
Guten Tag! Natürlich, wir liefern auch in die Schweiz.

Die Versandkosten sind um 5€ extra beim Bestellen. Das entspricht auch dem Versand in andere Länder.

→ 43 tok | 4.4s | 9.8 tok/s | 25 words

[q02] Koennten Sie mir bitte mitteilen, ob beim Versand eine Geschenkverpackung mit Karte moeglich ist.
Natürlich, das koennen Sie! Eine Geschenkverpackung mit Karte ist ganz moeglich. Sie koennen die Karte auch an die Empfängerin senden oder an einen anderen Empfänger setzen.

→ 49 tok | 4.7s | 10.4 tok/s | 26 words

[q03] Ich ziehe in zwei Wochen um und wuerde meine Bestellung erst danach erhalten wollen. Gibt es eine Moeglichkeit ein Wunschdatum fuer die Lieferung anzugeben? Andernfalls warte ich mit dem Bestellen bis nach dem Umzug.
Natürlich! Das koennen Sie ganz einfach selbst a

## 4. Save Results

Saves the raw generation results and VRAM measurements as JSON files to the Kaggle working directory.

Switch `if False` to `if True` to activate. The files can then be downloaded from the Kaggle output panel and committed to `results/` in the repository for Notebook 05.

In [11]:
if True:
    os.makedirs("/kaggle/working/results", exist_ok=True)

    files = {
        "raw_generations_base.json":    base_results,
        "raw_generations_teacher.json": teacher_results,
        "raw_generations_student.json": student_results,
        "vram_measurements.json":       vram_measurements,
    }

    for filename, data in files.items():
        path = f"/kaggle/working/results/{filename}"
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Saved: {path}  ({len(data)} entries)")

Saved: /kaggle/working/results/raw_generations_base.json  (60 entries)
Saved: /kaggle/working/results/raw_generations_teacher.json  (60 entries)
Saved: /kaggle/working/results/raw_generations_student.json  (60 entries)
Saved: /kaggle/working/results/vram_measurements.json  (3 entries)
